# LC 208 — Implement Trie (Prefix Tree)
**Day 64 | Pattern: Tries**

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px">
<strong>Core Insight:</strong> A Trie stores strings character-by-character
in a tree. Each node branches on one letter. Shared prefixes share nodes,
making prefix queries O(len) instead of O(n * len).
</div>

## Official Problem Statement

A **trie** (pronounced as "try") or **prefix tree** is a tree data structure
used to efficiently store and retrieve keys in a dataset of strings.
There are various applications of this data structure, such as autocomplete
and spell checker.

Implement the `Trie` class:
- `Trie()` Initializes the trie object.
- `void insert(String word)` Inserts the string `word` into the trie.
- `boolean search(String word)` Returns `true` if the string `word` is in
  the trie (i.e., was inserted before), and `false` otherwise.
- `boolean startsWith(String prefix)` Returns `true` if there is a
  previously inserted string that has the prefix `prefix`, and `false`
  otherwise.

**Constraints:**
- `1 <= word.length, prefix.length <= 2000`
- `word` and `prefix` consist only of lowercase English letters.
- At most `3 * 10^4` calls total to `insert`, `search`, and `startsWith`.

## What This Is Actually Asking

Build a tree where every character of a word occupies one node level.
Words that share a prefix share the same path from the root down.
A special `is_end` flag marks nodes where a complete word finishes.
`search` must reach the last char AND see `is_end=True`.
`startsWith` only needs to reach the last char of the prefix — no end flag
required.

## Walk Through an Example by Hand

Insert `"app"`, `"apple"`, `"apply"`:

```
Step 1 — insert("app")
  root -> 'a' -> 'p' -> 'p'[END]

Step 2 — insert("apple")
  root -> 'a' -> 'p' -> 'p'[END] -> 'l' -> 'e'[END]

Step 3 — insert("apply")
  root -> 'a' -> 'p' -> 'p'[END] -> 'l' -> 'e'[END]
                                          -> 'y'[END]

search("app")    => walk a->p->p, is_end=True  => True
search("ap")     => walk a->p,   is_end=False  => False
startsWith("ap") => walk a->p,   path exists   => True
search("appz")   => walk a->p->p, no 'z' child => False
```

## The Picture

```
                    root
                     |
                    [a]
                     |
                    [p]
                     |
                   [p] *END*          <- "app" ends here
                     |
                    [l]
                   /   \
                 [e]   [y]            <- "apple", "apply"
                *END*  *END*

  Each node:
  +---------------------------+
  | children: dict[char->node]|
  | is_end:   bool            |
  +---------------------------+

  insert  -> walk + create nodes + set is_end
  search  -> walk + check is_end at last node
  starts  -> walk + return True if path exists
```

## When To Use This Pattern

- When you need fast prefix lookups across many strings, think Trie.
- When autocomplete / type-ahead search is required, think Trie.
- When multiple strings share common prefixes and memory matters,
  think Trie (sharing prefix nodes).
- When you need to count words with a given prefix in O(len), think Trie.
- When spell-check or dictionary word validation is needed, think Trie.

## The Approach

Create a `TrieNode` with a `children` dictionary and an `is_end` boolean.
For `insert`, walk character by character, creating missing nodes, then set
`is_end=True` on the final node.
For `search`, walk the same way — return `False` immediately if a character
is missing, otherwise return the `is_end` flag of the last node.
For `startsWith`, walk just like `search` but return `True` as long as the
entire prefix path exists, ignoring `is_end`.

In [ ]:
# No external imports needed for Trie — uses built-in dict
from typing import Optional

In [ ]:
# ── Test Harness (class-based op-replay) ─────────────────────────────────

def run_trie_tests(TrieClass):
    """
    Replays operation sequences against TrieClass.
    Prints PASSED / FAILED per case + summary.
    """
    cases = [
        {
            "desc": "LC example — apple/app",
            "ops": [
                ("insert",     "apple",  None),
                ("insert",     "app",    None),
                ("search",     "apple",  True),
                ("search",     "app",    True),
                ("search",     "ap",     False),
                ("startsWith", "app",    True),
                ("startsWith", "apz",    False),
            ],
        },
        {
            "desc": "search miss before insert",
            "ops": [
                ("search",     "hello",  False),
                ("insert",     "hello",  None),
                ("search",     "hello",  True),
                ("search",     "hell",   False),
                ("startsWith", "hell",   True),
            ],
        },
        {
            "desc": "prefix equals full word",
            "ops": [
                ("insert",     "a",      None),
                ("search",     "a",      True),
                ("startsWith", "a",      True),
                ("search",     "ab",     False),
            ],
        },
    ]

    passed = failed = 0
    for case in cases:
        trie = TrieClass()
        case_ok = True
        for op, arg, expected in case["ops"]:
            if op == "insert":
                trie.insert(arg)
                result = None
            elif op == "search":
                result = trie.search(arg)
            else:
                result = trie.startsWith(arg)

            if expected is not None and result != expected:
                print(
                    f"  FAIL  {op}({arg!r}): "
                    f"got {result}, want {expected}"
                )
                case_ok = False

        label = "PASSED" if case_ok else "FAILED"
        print(f"[{label}] {case['desc']}")
        if case_ok:
            passed += 1
        else:
            failed += 1

    total = passed + failed
    print(f"\nResult: {passed}/{total} passed")


# ── Smoke-test with a stub that always returns False ─────────────────────
class _StubTrie:
    def insert(self, w): pass
    def search(self, w): return False
    def startsWith(self, p): return False

print("--- Harness smoke test (stub — expect failures) ---")
run_trie_tests(_StubTrie)

In [ ]:
# ── Solution Shell ────────────────────────────────────────────────────────

class TrieNode:
    """
    Single node in the Trie.

    Attributes
    ----------
    children : dict[str, TrieNode]
        Maps each outgoing character to the next node.
    is_end : bool
        True if a complete word ends at this node.
    """
    def __init__(self):
        self.children: dict = {}
        self.is_end: bool = False


class Trie:
    """
    LC 208 — Implement Trie (Prefix Tree)

    Design
    ------
    Root is an empty TrieNode.  Each insert/search/startsWith
    walks the tree one character at a time.

    Time  : O(L) per operation, L = word/prefix length
    Space : O(N * L) total nodes across all inserted words
    """

    def __init__(self):
        # TODO: initialise root node
        pass

    def insert(self, word: str) -> None:
        """
        Walk from root, creating TrieNode children as needed.
        Set is_end=True on the final node.

        Parameters
        ----------
        word : str  (lowercase letters only)
        """
        # TODO: implement
        print(f"[DEBUG] insert({word!r})")
        pass

    def search(self, word: str) -> bool:
        """
        Walk from root following each character.
        Return False if any char is missing.
        Return node.is_end at the last character.

        Parameters
        ----------
        word : str

        Returns
        -------
        bool
        """
        # TODO: implement
        print(f"[DEBUG] search({word!r})")
        pass

    def startsWith(self, prefix: str) -> bool:
        """
        Walk from root following each character of prefix.
        Return False if any char is missing.
        Return True if the full prefix path exists.

        Parameters
        ----------
        prefix : str

        Returns
        -------
        bool
        """
        # TODO: implement
        print(f"[DEBUG] startsWith({prefix!r})")
        pass

In [ ]:
# Uncomment and run when solution is ready
# run_trie_tests(Trie)

## Complexity

| Approach | Time (per op) | Space |
|---|---|---|
| Brute force (list of strings) | O(N * L) search | O(N * L) |
| Hash set | O(L) average | O(N * L) |
| **Trie (optimal)** | **O(L)** worst-case | **O(N * L)** nodes |

Where **N** = number of words, **L** = average word length.

Trie matches hash set on time but additionally supports O(L) prefix
queries — something a hash set cannot do efficiently.

## Real World Connection

At **Citi**, trading systems use trie-like structures to resolve ticker
symbol prefixes during order entry — allowing partial matches across
tens of thousands of instruments in sub-millisecond time.
In **AWS**, services like CloudWatch Logs use prefix-indexed namespaces
so customers can filter log groups by prefix without scanning all entries.
For **Data Engineering**, ETL pipelines sometimes maintain a trie of known
column name patterns to quickly classify incoming schema fields during
schema inference or data-quality validation.
Autocomplete in SQL editors (e.g., Redshift Query Editor) also leverages
tries to suggest table and column names as you type.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra